<a href="https://colab.research.google.com/github/clee2026/MSDS_498_Capstone/blob/main/nlp_categories/01_nyc311_nlp_issue_category_discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0. Install packages
!pip install -q pyarrow pandas scikit-learn sentence-transformers tqdm

In [ ]:
# 1. Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Configuration
# Change these paths and settings as needed.

import os
from pathlib import Path

PARQUET_PATH = "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_FINAL_UPDATED.parquet"

OUTPUT_DIR = "/content/nlp_issue_outputs"
# OUTPUT_DIR = "/content/drive/MyDrive/project_data/Capstone Course Project Files/outputs_nlp_issue_categories"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

MAX_ROWS_TO_SAMPLE = 300_000
PARQUET_BATCH_SIZE = 50_000

# Candidate text columns.
# The notebook will only use the columns that actually exist in the parquet.
CANDIDATE_TEXT_COLUMNS = [
    "complaint_type",
    "descriptor",
    "resolution_description",
    "incident_zip",
    "borough",
    "agency_name",
    "location_type"
]

# Best starting value. You can rerun with 10, 15, 20, 25, etc.
NUM_CLUSTERS = 20

RANDOM_STATE = 42

In [ ]:
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Inspect parquet schema

import pyarrow.parquet as pq
import pandas as pd

pf = pq.ParquetFile(PARQUET_PATH)
parquet_columns = pf.schema.names

print("Number of parquet columns:", len(parquet_columns))
print("First 50 columns:")
print(parquet_columns[:50])

available_text_columns = [c for c in CANDIDATE_TEXT_COLUMNS if c in parquet_columns]

print("\nAvailable candidate text columns:")
print(available_text_columns)

if not available_text_columns:
    raise ValueError("None of the candidate text columns were found. Update CANDIDATE_TEXT_COLUMNS after reviewing parquet_columns.")

In [ ]:
# 4. Load a sample from the 20M record parquet

from tqdm.auto import tqdm

dfs = []
rows_loaded = 0

for batch in tqdm(pf.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=available_text_columns)):
    df_batch = batch.to_pandas()
    dfs.append(df_batch)
    rows_loaded += len(df_batch)

    if rows_loaded >= MAX_ROWS_TO_SAMPLE:
        break

df = pd.concat(dfs, ignore_index=True).head(MAX_ROWS_TO_SAMPLE)

print("Loaded sample shape:", df.shape)
display(df.head())

In [ ]:
# 5. Build combined text field

# Heavier weight is given to complaint_type and descriptor because those fields usually
# describe the issue more directly than resolution_description.

def safe_col(name):
    if name in df.columns:
        return df[name].fillna("").astype(str)
    return ""

combined_parts = []

if "complaint_type" in df.columns:
    combined_parts.append(safe_col("complaint_type"))
if "descriptor" in df.columns:
    combined_parts.append(safe_col("descriptor"))
if "resolution_description" in df.columns:
    combined_parts.append(safe_col("resolution_description"))

# Optional context fields. These should not dominate the category, but can help interpret issues.
for optional_col in ["agency_name", "location_type", "borough"]:
    if optional_col in df.columns:
        combined_parts.append(safe_col(optional_col))

df["combined_text"] = combined_parts[0]
for part in combined_parts[1:]:
    df["combined_text"] = df["combined_text"] + " " + part

df["combined_text"] = df["combined_text"].str.strip()
df = df[df["combined_text"] != ""].copy()

print("Rows with usable text:", df.shape[0])
display(df[available_text_columns + ["combined_text"]].head())

In [ ]:
# 6. Clean text

import re

DOMAIN_STOPWORDS = {
    "please", "call", "called", "complaint", "customer", "service",
    "request", "reported", "information", "provided", "will", "may",
    "nyc", "city", "department"
}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if len(t) > 2 and t not in DOMAIN_STOPWORDS]
    return " ".join(tokens)

df["clean_text"] = df["combined_text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 5].copy()

print("Rows after cleaning:", df.shape[0])
display(df[["combined_text", "clean_text"]].head())

In [ ]:
# 7. Create sentence embeddings

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

embeddings = model.encode(
    df["clean_text"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

In [ ]:
# 8. Cluster issue descriptions

from sklearn.cluster import MiniBatchKMeans

kmeans = MiniBatchKMeans(
    n_clusters=NUM_CLUSTERS,
    random_state=RANDOM_STATE,
    batch_size=4096,
    n_init="auto"
)

df["issue_cluster"] = kmeans.fit_predict(embeddings)

print(df["issue_cluster"].value_counts().sort_index())

In [ ]:
# 9. Summarize clusters with top terms and example records

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=10,
    max_df=0.80,
    ngram_range=(1, 2)
)

X_tfidf = vectorizer.fit_transform(df["clean_text"])
terms = np.array(vectorizer.get_feature_names_out())

summaries = []

for cluster_id in sorted(df["issue_cluster"].unique()):
    idx = df["issue_cluster"].values == cluster_id
    cluster_size = int(idx.sum())

    mean_tfidf = X_tfidf[idx].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-12:][::-1]
    top_terms = terms[top_idx].tolist()

    examples = df.loc[idx, "combined_text"].dropna().astype(str).head(5).tolist()

    summaries.append({
        "issue_cluster": cluster_id,
        "record_count": cluster_size,
        "share_of_sample": cluster_size / len(df),
        "top_terms": ", ".join(top_terms),
        "example_1": examples[0] if len(examples) > 0 else "",
        "example_2": examples[1] if len(examples) > 1 else "",
        "suggested_business_label": ""
    })

summary_df = pd.DataFrame(summaries).sort_values("record_count", ascending=False)

display(summary_df)

In [ ]:
# 10. Optional helper labels
# These are starting labels only. Review and edit the CSV manually.

def suggest_label(top_terms):
    text = top_terms.lower()

    rules = [
        ("Heating or hot water issue", ["heat", "hot water", "boiler", "radiator"]),
        ("Noise complaint", ["noise", "loud", "music", "party"]),
        ("Illegal parking or blocked access", ["parking", "blocked", "driveway", "vehicle"]),
        ("Street or sidewalk condition", ["street", "sidewalk", "pothole", "curb"]),
        ("Sanitation or trash issue", ["trash", "garbage", "sanitation", "dirty"]),
        ("Water leak or plumbing issue", ["leak", "water", "plumbing", "sewer"]),
        ("Rodent or pest issue", ["rodent", "rat", "mice", "pest"]),
        ("Building or housing maintenance", ["building", "apartment", "landlord", "maintenance"]),
        ("Tree or park issue", ["tree", "park", "branch"]),
        ("Traffic or signal issue", ["traffic", "signal", "light"])
    ]

    for label, keywords in rules:
        if any(k in text for k in keywords):
            return label

    return "Needs manual review"

summary_df["suggested_business_label"] = summary_df["top_terms"].apply(suggest_label)

display(summary_df)

In [ ]:
import os
import joblib
import json

sample_output_path = os.path.join(OUTPUT_DIR, "nlp_issue_category_sample.parquet")
summary_output_path = os.path.join(OUTPUT_DIR, "nlp_issue_cluster_summary.csv")
model_output_path = os.path.join(OUTPUT_DIR, "nlp_issue_kmeans_model.joblib")
config_output_path = os.path.join(OUTPUT_DIR, "nlp_issue_category_config.json")

df.to_parquet(sample_output_path, index=False)
summary_df.to_csv(summary_output_path, index=False)
joblib.dump(kmeans, model_output_path)

config = {
    "available_text_columns": available_text_columns,
    "num_clusters": NUM_CLUSTERS,
    "embedding_model_name": EMBEDDING_MODEL_NAME
}

with open(config_output_path, "w") as f:
    json.dump(config, f, indent=2)

print("Saved to:", OUTPUT_DIR)

In [ ]:
# ZIP ALL OUTPUTS FOR DOWNLOAD


import shutil

ZIP_PATH = "/content/nlp_issue_outputs.zip"

shutil.make_archive(
    base_name=ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=OUTPUT_DIR
)

print("ZIP created at:", ZIP_PATH)